# Gene prediction using the Enformer architecture for coding (CDS) and non-coding region prediction

In [1]:

from IPython import get_ipython
get_ipython().magic('reset -sf')


In [2]:
import torch
torch.cuda.empty_cache()

## Step 1: Data Preprocessing

In [3]:
from Bio import SeqIO
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from torch.utils.data import Subset
from torch.utils.data import DataLoader, Dataset
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except Exception:
    plt = None
    sns = None
from sklearn.metrics import precision_recall_curve, average_precision_score

### Parsing GFF3 and FASTA

In [4]:
file_path = "data/ergobibamus_cyprinoides_labels.gff3"

# Specify header=0 to use the first row as column names
gff3_df = pd.read_csv(file_path, sep='\t', comment='#', header=0)

pd.DataFrame(gff3_df)


In [5]:
def parse_gff3(gff3_file):
    data = []
    with open(gff3_file, 'r') as file:
        for line in file:
            if line.startswith("#") or line.startswith("seq_id"):
                continue
            fields = line.strip().split('\t')
            if len(fields) == 9:  # Ensure valid GFF3 format
                data.append({
                    'seq_id': fields[0],
                    'source': fields[1],
                    'type': fields[2],
                    'start': int(fields[3]),
                    'end': int(fields[4]),
                    'score': fields[5],
                    'strand': fields[6],
                    'phase': fields[7],
                    'attributes': fields[8],
                })
    return pd.DataFrame(data)

def parse_fasta(fasta_file):
    sequences = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences[record.id] = str(record.seq)
    return sequences

# Example
# gff3_file = "data/extracted_genomes/merged.gff3"
# fasta_file = "data/extracted_genomes/merged.fasta"

gff3_file = "data/ergobibamus_cyprinoides_labels.gff3"
fasta_file = "data/ergobibamus_cyprinoides_genome.fasta"

gff3_df = parse_gff3(gff3_file)
sequences = parse_fasta(fasta_file)
gff3_df.head()

In [6]:
sequences['tig00000491']

In [7]:
len(sequences['tig00000491'])

### Generating Training Samples
Create sliding windows from the genome sequences and assign labels based on gff3.

In [8]:
def generate_samples(sequences, gff3_df, window_size=1000):
    features = []
    labels = []

    # Wrap the outer loop with tqdm to monitor progress
    for seq_id, sequence in tqdm(sequences.items(), desc="Processing sequences", unit="sequence"):
        sequence_length = len(sequence)
        annotation = gff3_df[gff3_df['seq_id'] == seq_id]

        # Wrap the inner loop with tqdm to monitor window progress for each sequence
        for start in tqdm(range(0, sequence_length - window_size + 1, window_size), desc=f"Processing windows for {seq_id}", leave=False, unit="window"):
            end = start + window_size
            subseq = sequence[start:end]

            # Binary label: coding (1) or non-coding (0)
            label = any(
                (annotation['start'] <= end) & (annotation['end'] >= start) &
                (annotation['type'].isin(['start_codon', 'stop_codon','exon']))
            )
            features.append(subseq)
            labels.append(int(label))

    return features, labels

window_size = 1000
features, labels = generate_samples(sequences, gff3_df, window_size=window_size)

Processing windows for tig00000079:   0%|          | 0/150 [00:00<?, ?window/s]
                                                                               
Processing windows for tig00000031:   0%|          | 0/137 [00:00<?, ?window/s]
                                                                               
Processing windows for tig00000513:   0%|          | 0/96 [00:00<?, ?window/s]
                                                                              
Processing windows for tig00000510:   0%|          | 0/95 [00:00<?, ?window/s]
                                                                              
Processing windows for tig00000506:   0%|          | 0/89 [00:00<?, ?window/s]
                                                                              
Processing windows for tig00000491:   0%|          | 0/78 [00:00<?, ?window/s]
                                                                              
Processing windows for tig00000445:   0%|       

In [9]:
len(features), len(labels)

In [10]:
from collections import Counter

label_counts = Counter(labels)
print("Label counts:", label_counts)

Label counts: Counter({1: 12331, 0: 2077})


In [11]:
class_0_indices = [i for i, label in enumerate(labels) if label == 0]
class_1_indices = [i for i, label in enumerate(labels) if label == 1]

# Balance classes by undersampling the majority class only
if len(class_0_indices) > len(class_1_indices):
    majority_indices, minority_indices = class_0_indices, class_1_indices
elif len(class_1_indices) > len(class_0_indices):
    majority_indices, minority_indices = class_1_indices, class_0_indices
else:
    majority_indices, minority_indices = class_0_indices, class_1_indices

if len(majority_indices) != len(minority_indices):
    majority_sampled = resample(
        majority_indices,
        replace=False,
        n_samples=len(minority_indices),
        random_state=42,
    )
else:
    majority_sampled = majority_indices

balanced_indices = majority_sampled + minority_indices
features = [features[i] for i in balanced_indices]
labels = [labels[i] for i in balanced_indices]


In [12]:
label_counts = Counter(labels)
print("Label counts:", label_counts)

Label counts: Counter({1: 2077, 0: 2077})


### Encoding DNA Sequences
DNA sequences are encoded into a numerical format for the model.

In [13]:
def encode_sequence(sequence):
    encoding = {'A': 0, 'C': 1, 'G': 2, 'T': 3, 'N': 4}
    return [encoding.get(base, 4) for base in sequence]

encoded_features = [encode_sequence(seq) for seq in features]

In [14]:
unique_values = np.unique([val for seq in encoded_features for val in seq])
print("Unique values in encoded_features:", unique_values)

Unique values in encoded_features: [0 1 2 3 4]


In [15]:
lengths = [len(seq) for seq in encoded_features]
print("Lengths of sequences:", set(lengths))

Lengths of sequences: {1000}


## Step 2: Transformer Model Architecture


In [16]:
class GenePredictionModel(nn.Module):
    def __init__(self, input_length, num_classes, embed_dim=256, num_heads=4, num_layers=4):
        super(GenePredictionModel, self).__init__()
        self.embedding = nn.Embedding(5, embed_dim)  # A, C, G, T, N

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            activation='relu',
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(1, 0, 2)  # Sequence-first for transformer
        x = self.transformer(x)
        x = x.mean(dim=0)  # Global average pooling
        x = self.fc(x)
        return x

## Step 3: Training and Validation Functions

In [17]:
def train_one_epoch(model, optimizer, criterion, dataloader, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    # for DataLoader with tqdm
    for batch in tqdm(dataloader, desc="Training", unit="batch"):
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return running_loss / len(dataloader), acc

def validate(model, criterion, dataloader, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    # for DataLoader with tqdm
    with torch.no_grad(): # Disabling Dropout and BatchNorm
        for batch in tqdm(dataloader, desc="Validation", unit="batch"):
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return running_loss / len(dataloader), acc

## Step 4: DataLoader and Training Script

In [18]:
class DNASequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = [torch.tensor(seq, dtype=torch.long) for seq in sequences]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [19]:
len(encoded_features), len(labels)

In [20]:
# Split data into train (80%) and temp (20% for validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(encoded_features, labels, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Prepare datasets
train_dataset = DNASequenceDataset(X_train, y_train)
val_dataset = DNASequenceDataset(X_val, y_val)
test_dataset = DNASequenceDataset(X_test, y_test)

# Prepare dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Training script
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GenePredictionModel(input_length=window_size, num_classes=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

/Users/zahratavakolirad/Desktop/Zahra/Dal/fall-2024/ML/.venv/lib/python3.9/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


#### Note: Choosing a Data Subset in the Absence of GPU Resources
- Code to Create Mini-Loader

In [21]:
def create_mini_loader(dataset, subset_size, batch_size=32, shuffle=True):

    subset_indices = torch.randperm(len(dataset))[:subset_size]
    subset = Subset(dataset, subset_indices)

    # Create a DataLoader for the subset
    mini_loader = DataLoader(subset, batch_size=batch_size, shuffle=shuffle)
    return mini_loader

In [22]:
# Create mini-loaders for training and validation subsets
mini_train_loader = create_mini_loader(train_dataset, subset_size=1000, batch_size=16, shuffle=True)
mini_val_loader = create_mini_loader(val_dataset, subset_size=500, batch_size=16, shuffle=False)
mini_test_loader = create_mini_loader(test_dataset, subset_size=500, batch_size=16, shuffle=False)

# Check the number of batches in the mini-loaders
print(f"Mini train loader batches: {len(mini_train_loader)}")
print(f"Mini val loader batches: {len(mini_val_loader)}")
print(f"Mini Test loader batches: {len(mini_test_loader)}")

# Example training loop using mini-loader
for batch in mini_train_loader:
    inputs, labels = batch
    print("Inputs shape:", inputs.shape)
    print("Labels shape:", labels.shape)
    break

Mini train loader batches: 63
Mini val loader batches: 26
Mini Test loader batches: 26
Inputs shape: torch.Size([16, 1000])
Labels shape: torch.Size([16])


In [23]:
epochs = 10
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, optimizer, criterion, mini_train_loader, device)
    val_loss, val_acc = validate(model, criterion, mini_val_loader, device)
    print(f"Epoch {epoch+1}/{epochs}: Train Loss: {train_loss:.3f}, Train Acc: {train_acc:.3f}, Val Loss: {val_loss:.3f}, Val Acc: {val_acc:.3f}")

Validation: 100%|##########| 26/26 [00:08<00:00,  3.09batch/s]
Epoch 1/10: Train Loss: 0.820, Train Acc: 0.508, Val Loss: 0.690, Val Acc: 0.540
Validation: 100%|##########| 26/26 [00:09<00:00,  2.75batch/s]
Epoch 2/10: Train Loss: 0.708, Train Acc: 0.492, Val Loss: 0.699, Val Acc: 0.540
Validation: 100%|##########| 26/26 [00:09<00:00,  2.74batch/s]
Epoch 3/10: Train Loss: 0.690, Train Acc: 0.541, Val Loss: 0.640, Val Acc: 0.720
Validation: 100%|##########| 26/26 [00:12<00:00,  2.01batch/s]
Epoch 4/10: Train Loss: 0.712, Train Acc: 0.693, Val Loss: 0.751, Val Acc: 0.460
Validation: 100%|##########| 26/26 [00:09<00:00,  2.76batch/s]
Epoch 5/10: Train Loss: 0.729, Train Acc: 0.482, Val Loss: 0.746, Val Acc: 0.460
Validation: 100%|##########| 26/26 [00:09<00:00,  2.64batch/s]
Epoch 6/10: Train Loss: 0.699, Train Acc: 0.484, Val Loss: 0.691, Val Acc: 0.540
Validation: 100%|##########| 26/26 [00:10<00:00,  2.36batch/s]
Epoch 7/10: Train Loss: 0.700, Train Acc: 0.499, Val Loss: 0.743, Val Acc

### For all data training

In [24]:
# epochs = 10
# for epoch in range(epochs):
#     train_loss, train_acc = train_one_epoch(model, optimizer, criterion, train_loader, device)
#     val_loss, val_acc = validate(model, criterion, val_loader, device)
#     print(f"Epoch {epoch+1}/{epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

## Step 5: Metrics and Final Evaluation
### Metrics Implementation
We use sklearn.metrics to compute metrics such as accuracy, precision, recall, and F1-score. These metrics are critical for understanding the performance of the gene prediction model.

In [25]:
def train_one_epoch(model, optimizer, criterion, data_loader, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for inputs, labels in data_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(data_loader), correct / total

def validate(model, criterion, data_loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    y_true = []
    y_pred = []
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    return total_loss / len(data_loader), correct / total, y_true, y_pred


In [26]:
def compute_metrics(y_true, y_pred):

    report = classification_report(y_true, y_pred, target_names=["Non-coding", "Coding"])
    print("\nClassification Report:")
    print(report)

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    return report, cm
    report, cm = compute_metrics(y_true, y_pred)
    print(report)

### Final Evaluation Script
After training, you can use the following script to evaluate your model on a test dataset.

In [27]:
# Split data into train, validation, and test sets
# X_train, X_temp, y_train, y_temp = train_test_split(encoded_features, labels, test_size=0.3, random_state=42)
# X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Prepare datasets and dataloaders
# test_dataset = DNASequenceDataset(X_test, y_test)
# test_loader = DataLoader(test_dataset, batch_size=32)

mini_test_loader = create_mini_loader(test_dataset, subset_size=50, batch_size=16, shuffle=False)

# Evaluate on test set
def evaluate_on_test(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_labels, all_preds

# Compute predictions
y_true, y_pred = evaluate_on_test(model, mini_test_loader, device)

# Compute and display metrics
compute_metrics(y_true, y_pred)

/Users/zahratavakolirad/Desktop/Zahra/Dal/fall-2024/ML/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/zahratavakolirad/Desktop/Zahra/Dal/fall-2024/ML/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/zahratavakolirad/Desktop/Zahra/Dal/fall-2024/ML/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter t

## Step 6: Saving and Loading the Model
It’s essential to save your trained model for reuse or inference. You can also load a saved model for evaluation.

In [28]:
# Save the trained model
torch.save(model.state_dict(), "enformer_gene_prediction.pth")

# Load the trained model
model = GenePredictionModel(input_length=window_size, num_classes=2)
model.load_state_dict(torch.load("enformer_gene_prediction.pth"))
model.to(device)

/Users/zahratavakolirad/Desktop/Zahra/Dal/fall-2024/ML/.venv/lib/python3.9/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


### Step 6-1: Visualizing the Results
Visualizations can help better understand model performance.

In [ ]:
# Plot confusion matrix
def plot_confusion_matrix(cm, class_names):
    if plt is None or sns is None:
        print("matplotlib/seaborn not installed; skipping confusion matrix plot.")
        return
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()

# Visualize the confusion matrix
class_names = ["Non-coding", "Coding"]
plot_confusion_matrix(confusion_matrix(y_true, y_pred), class_names)


matplotlib/seaborn not installed; skipping confusion matrix plot.


### Step 6.2: Adding Precision-Recall Curve
The precision-recall curve provides additional insights, especially for imbalanced datasets.

In [30]:
def plot_precision_recall(y_true, y_scores):
    precision, recall, _ = precision_recall_curve(y_true, y_scores[:, 1])  # Assuming class 1 is coding
    avg_precision = average_precision_score(y_true, y_scores[:, 1])

    if plt is None:
        print("matplotlib not installed; skipping precision-recall plot.")
        return
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f"Avg Precision = {avg_precision:.2f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid()
    plt.show()

# Get raw scores for the test set
def get_scores(model, test_loader, device):
    model.eval()
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_scores.extend(outputs.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_scores)

# Compute and plot precision-recall curve
y_true, y_scores = get_scores(model, test_loader, device)
plot_precision_recall(y_true, y_scores)

matplotlib not installed; skipping precision-recall plot.
